# Treino Baseline V4 — Comparativo

**Projeto:** CheckAI — Classificador binário de fake news em PT-BR

Treina e compara modelos clássicos TF-IDF nos datasets:
- V3 balanced (baseline histórico)
- V4 balanced (base expandida com CheckAI Autoral + GFC)
- V4 text_control (ablação de viés de tamanho)

**Objetivo principal:** verificar se a expansão da base própria melhora a generalização.

> Nenhum modelo em produção é alterado. V1/V2/V3/V4 não são modificados.

In [ ]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'src' else _cwd
DADOS_DIR  = PROJECT_ROOT / 'dados' / 'dataset_unificado' / 'final'
MODELOS_DIR = PROJECT_ROOT / 'modelos'
MODELOS_DIR.mkdir(exist_ok=True)

SEED      = 42
TEST_SIZE = 0.2
TS        = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'TIMESTAMP    : {TS}')

## Seção 1 — Detecção automática de datasets

In [ ]:
def latest_match(pattern: str) -> Path:
    files = [p for p in DADOS_DIR.glob(pattern) if p.stat().st_size > 0]
    if not files:
        raise FileNotFoundError(f'Nenhum arquivo encontrado: {pattern}')
    return max(files, key=lambda p: p.stat().st_mtime)

PATH_V3_BALANCED          = latest_match('dataset_final_treino_v3_balanced_*.csv')
PATH_V4_BALANCED          = latest_match('dataset_final_treino_v4_balanced_*.csv')
PATH_V4_TEXT_CONTROL      = latest_match('dataset_final_treino_v4_text_control_*.csv')
PATH_V4_SC_BALANCED       = latest_match('dataset_final_treino_v4_source_control_balanced_*.csv')
PATH_V4_SC_TEXT_CONTROL   = latest_match('dataset_final_treino_v4_source_control_text_control_*.csv')

print('Datasets detectados:')
print(f'  V3 balanced          : {PATH_V3_BALANCED.name}')
print(f'  V4 balanced          : {PATH_V4_BALANCED.name}')
print(f'  V4 text_control      : {PATH_V4_TEXT_CONTROL.name}')
print(f'  V4 SC balanced       : {PATH_V4_SC_BALANCED.name}')
print(f'  V4 SC text_control   : {PATH_V4_SC_TEXT_CONTROL.name}')

## Seção 2 — Funções auxiliares e carregamento de datasets

In [ ]:
def carregar_dataset(path: Path, nome: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
    print(f'\n{"-"*60}')
    print(f'Dataset : {nome}')
    print(f'Arquivo : {path.name}')
    print(f'Shape   : {df.shape}')

    # Texto preferencial
    if 'texto_principal_modelo' in df.columns:
        df['texto_treino'] = df['texto_principal_modelo'].fillna('').astype(str)
        print('Campo texto: texto_principal_modelo')
    else:
        df['texto_treino'] = df['texto_principal'].fillna('').astype(str)
        print('Campo texto: texto_principal (fallback)')

    # Normalizar label -> int
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    antes = len(df)
    df = df[df['label'].isin([0, 1])].copy()
    df['label'] = df['label'].astype(int)
    if len(df) < antes:
        print(f'Labels inválidos removidos: {antes - len(df)}')

    # Remover texto vazio
    df = df[df['texto_treino'].str.strip() != ''].copy()

    print(f'Final   : {len(df)} registros')
    print(f'label=0 : {(df["label"]==0).sum()}  |  label=1 : {(df["label"]==1).sum()}')
    if 'dataset_origem' in df.columns:
        print(f'dataset_origem: {df["dataset_origem"].value_counts().to_dict()}')
    return df.reset_index(drop=True)

In [ ]:
df_v3_balanced         = carregar_dataset(PATH_V3_BALANCED,        'V3 balanced')
df_v4_balanced         = carregar_dataset(PATH_V4_BALANCED,        'V4 balanced')
df_v4_text_control     = carregar_dataset(PATH_V4_TEXT_CONTROL,    'V4 text_control')
df_v4_sc_balanced      = carregar_dataset(PATH_V4_SC_BALANCED,     'V4 SC balanced')
df_v4_sc_text_control  = carregar_dataset(PATH_V4_SC_TEXT_CONTROL, 'V4 SC text_control')

DATASETS = {
    'v3_balanced':          {'df': df_v3_balanced,        'path': PATH_V3_BALANCED},
    'v4_balanced':          {'df': df_v4_balanced,        'path': PATH_V4_BALANCED},
    'v4_text_control':      {'df': df_v4_text_control,    'path': PATH_V4_TEXT_CONTROL},
    'v4_sc_balanced':       {'df': df_v4_sc_balanced,     'path': PATH_V4_SC_BALANCED},
    'v4_sc_text_control':   {'df': df_v4_sc_text_control, 'path': PATH_V4_SC_TEXT_CONTROL},
}

## Seção 3 — Diagnóstico pré-treino

In [ ]:
def diagnosticar(df: pd.DataFrame, nome: str):
    SEP = '=' * 65
    print(f'\n{SEP}')
    print(f'DIAGNÓSTICO — {nome}')
    print(SEP)
    print(f'Total registros : {len(df)}')
    print(f'label=0         : {(df["label"]==0).sum()}')
    print(f'label=1         : {(df["label"]==1).sum()}')
    _lens = df['texto_treino'].str.len()
    print(f'chars média     : {_lens.mean():.0f}  mediana: {_lens.median():.0f}')
    print(f'chars min/max   : {_lens.min()} / {_lens.max()}')
    if 'dataset_origem' in df.columns:
        print(f'\ndataset_origem:')
        for orig, cnt in df['dataset_origem'].value_counts().items():
            print(f'  {str(orig):<30} {cnt:>6}  ({100*cnt/len(df):.1f}%)')
    if 'origem_qualidade' in df.columns:
        print(f'\norigem_qualidade:')
        for q, cnt in df['origem_qualidade'].value_counts().items():
            print(f'  {str(q):<30} {cnt:>6}')

for nome, info in DATASETS.items():
    diagnosticar(info['df'], nome)

## Seção 4 — Split treino/teste estratificado

In [ ]:
def fazer_split(df: pd.DataFrame, nome: str) -> dict:
    X   = df['texto_treino'].astype(str).values
    y   = df['label'].values
    idx = df.index.values

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, idx, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    print(f'[{nome}] treino={len(X_train)}  teste={len(X_test)}  '
          f'label_test=0:{(y_test==0).sum()} / 1:{(y_test==1).sum()}')
    return dict(X_train=X_train, X_test=X_test,
                y_train=y_train, y_test=y_test,
                idx_train=idx_train, idx_test=idx_test)

for nome, info in DATASETS.items():
    info['split'] = fazer_split(info['df'], nome)

## Seção 5 — Pipelines TF-IDF

- `analyzer=word`, `ngram_range=(1,2)`, `min_df=2`, `max_df=0.9`,
  `max_features=50000`, `sublinear_tf=True`
- **LogReg:** solver=lbfgs, class_weight=balanced
- **SVM:** LinearSVC + CalibratedClassifierCV, class_weight=balanced
- **NB:** MultinomialNB (sublinear_tf=False)

In [ ]:
TFIDF_PARAMS = dict(
    lowercase=True,
    strip_accents='unicode',
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    max_features=50000,
    sublinear_tf=True,
)
TFIDF_PARAMS_NB = {**TFIDF_PARAMS, 'sublinear_tf': False}


def make_logreg() -> Pipeline:
    return Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LogisticRegression(
            solver='lbfgs',
            class_weight='balanced',
            max_iter=1000,
            random_state=SEED,
        )),
    ])


def make_svm() -> Pipeline:
    return Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   CalibratedClassifierCV(
            LinearSVC(
                class_weight='balanced',
                max_iter=5000,
                random_state=SEED,
            )
        )),
    ])


def make_nb() -> Pipeline:
    return Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS_NB)),
        ('clf',   MultinomialNB()),
    ])


MODELOS_FACTORY = {
    'logreg': make_logreg,
    'svm':    make_svm,
    'nb':     make_nb,
}

print('Pipelines definidos: logreg (lbfgs) | svm (CalibratedSVC) | nb (MultinomialNB)')

## Seção 6 — Treinamento (3 datasets × 3 modelos)

In [ ]:
RESULTADOS = {}

print('Iniciando treinamento...\n')
for ds_nome, ds_info in DATASETS.items():
    split   = ds_info['split']
    X_train = split['X_train']
    y_train = split['y_train']

    for mod_nome, factory in MODELOS_FACTORY.items():
        chave = f'{ds_nome}__{mod_nome}'
        print(f'  Treinando {chave} ...')
        pipe = factory()
        pipe.fit(X_train, y_train)
        n_feat = pipe.named_steps['tfidf'].get_feature_names_out().shape[0]
        print(f'    features TF-IDF: {n_feat}')
        RESULTADOS[chave] = {
            'pipe':     pipe,
            'ds_nome':  ds_nome,
            'mod_nome': mod_nome,
            'n_feat':   n_feat,
        }

print(f'\nTreinamento concluído. {len(RESULTADOS)} modelos.')

## Seção 7 — Avaliação de modelos

In [ ]:
def avaliar(pipe: Pipeline, X_test, y_test) -> tuple:
    y_pred  = pipe.predict(X_test)
    y_score = None
    try:
        y_score = pipe.predict_proba(X_test)[:, 1]
    except Exception:
        try:
            y_score = pipe.decision_function(X_test)
        except Exception:
            pass

    m = {
        'accuracy':          round(accuracy_score(y_test, y_pred), 4),
        'precision_macro':   round(precision_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'recall_macro':      round(recall_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'f1_macro':          round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'precision_label_0': round(precision_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0), 4),
        'recall_label_0':    round(recall_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0), 4),
        'f1_label_0':        round(f1_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0), 4),
        'precision_label_1': round(precision_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0), 4),
        'recall_label_1':    round(recall_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0), 4),
        'f1_label_1':        round(f1_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0), 4),
        'roc_auc':           None,
    }
    if y_score is not None:
        try:
            m['roc_auc'] = round(roc_auc_score(y_test, y_score), 4)
        except Exception:
            pass
    return m, y_pred, y_score


def imprimir_avaliacao(chave: str, m: dict, y_test, y_pred):
    SEP = '-' * 60
    print(f'\n{SEP}')
    print(f'{chave}')
    print(SEP)
    print(f'  accuracy={m["accuracy"]}  f1_macro={m["f1_macro"]}  roc_auc={m["roc_auc"]}')
    print(f'  label=0 -> p={m["precision_label_0"]} r={m["recall_label_0"]} f1={m["f1_label_0"]}')
    print(f'  label=1 -> p={m["precision_label_1"]} r={m["recall_label_1"]} f1={m["f1_label_1"]}')
    cm = confusion_matrix(y_test, y_pred)
    print(f'  Matriz de confusão:\n{cm}')


print('Avaliando todos os modelos...\n')
for chave, res in RESULTADOS.items():
    ds_nome = res['ds_nome']
    split   = DATASETS[ds_nome]['split']
    m, y_pred, y_score = avaliar(res['pipe'], split['X_test'], split['y_test'])
    imprimir_avaliacao(chave, m, split['y_test'], y_pred)
    res['metricas'] = m
    res['y_pred']   = y_pred
    res['y_score']  = y_score

print('\nAvaliação concluída.')

## Seção 8 — Análise por dataset_origem

Para todos os modelos de V4 balanced. Foco em CHECKAI_AUTORAL e CHECKAI_PROPRIO_GFC.

In [ ]:
registros_por_origem = []

def analisar_por_origem(chave: str, res: dict):
    ds_nome = res['ds_nome']
    df      = DATASETS[ds_nome]['df']
    split   = DATASETS[ds_nome]['split']

    if 'dataset_origem' not in df.columns:
        return

    df_test           = df.loc[split['idx_test']].copy()
    df_test['_pred']  = res['y_pred']
    y_test            = split['y_test']

    print(f'\n{"-"*60}')
    print(f'Por dataset_origem — {chave}')
    print(f'{"-"*60}')

    for origem in sorted(df_test['dataset_origem'].dropna().unique()):
        sub  = df_test[df_test['dataset_origem'] == origem]
        if len(sub) < 5:
            continue
        yt   = sub['label'].values
        yp   = sub['_pred'].values
        f1   = round(f1_score(yt, yp, average='macro', zero_division=0), 4)
        acc  = round(accuracy_score(yt, yp), 4)
        n0   = (yt == 0).sum()
        n1   = (yt == 1).sum()
        print(f'  {str(origem):<30} n={len(sub):>5} (0:{n0} 1:{n1})  f1={f1}  acc={acc}')
        registros_por_origem.append({
            'chave': chave, 'dataset_treino': ds_nome, 'modelo': res['mod_nome'],
            'dataset_origem': origem, 'n_test': len(sub), 'n_label0': int(n0), 'n_label1': int(n1),
            'f1_macro': f1, 'accuracy': acc,
        })

# Rodar para V3, V4 balanced e V4 SC balanced (todos os modelos)
for chave, res in RESULTADOS.items():
    if 'v3_balanced' in chave or 'v4_balanced' in chave or 'v4_sc_balanced' in chave:
        analisar_por_origem(chave, res)

## Seção 9 — Análise por origem_qualidade

ROTULO_FORTE (GFC), ROTULO_ASSUMIDO_ALTO (Autoral), ROTULO_ACADEMICO (V3 herdado).

In [ ]:
registros_por_qualidade = []

def analisar_por_qualidade(chave: str, res: dict):
    ds_nome = res['ds_nome']
    df      = DATASETS[ds_nome]['df']
    split   = DATASETS[ds_nome]['split']

    if 'origem_qualidade' not in df.columns:
        return

    df_test           = df.loc[split['idx_test']].copy()
    df_test['_pred']  = res['y_pred']

    print(f'\n{"-"*60}')
    print(f'Por origem_qualidade — {chave}')
    print(f'{"-"*60}')

    for qual in sorted(df_test['origem_qualidade'].fillna('(vazio)').unique()):
        sub = df_test[df_test['origem_qualidade'].fillna('(vazio)') == qual]
        if len(sub) < 5:
            continue
        yt  = sub['label'].values
        yp  = sub['_pred'].values
        f1  = round(f1_score(yt, yp, average='macro', zero_division=0), 4)
        acc = round(accuracy_score(yt, yp), 4)
        print(f'  {str(qual):<28} n={len(sub):>5}  f1={f1}  acc={acc}')
        registros_por_qualidade.append({
            'chave': chave, 'dataset_treino': ds_nome, 'modelo': res['mod_nome'],
            'origem_qualidade': qual, 'n_test': len(sub),
            'f1_macro': f1, 'accuracy': acc,
        })

# Rodar para V3, V4 balanced e V4 SC balanced
for chave, res in RESULTADOS.items():
    if 'v3_balanced' in chave or 'v4_balanced' in chave or 'v4_sc_balanced' in chave:
        analisar_por_qualidade(chave, res)

## Seção 10 — Análise por faixa de tamanho

In [ ]:
registros_por_tamanho = []

def analisar_por_tamanho(chave: str, res: dict):
    ds_nome = res['ds_nome']
    df      = DATASETS[ds_nome]['df']
    split   = DATASETS[ds_nome]['split']

    col_faixa = next((c for c in ['faixa_tamanho_modelo','faixa_tamanho'] if c in df.columns), None)
    if col_faixa is None:
        return

    df_test          = df.loc[split['idx_test']].copy()
    df_test['_pred'] = res['y_pred']

    print(f'\n{"-"*60}')
    print(f'Por faixa_tamanho_modelo — {chave}')
    print(f'{"-"*60}')

    for faixa in ['curto', 'medio', 'longo', 'muito_longo']:
        sub = df_test[df_test[col_faixa] == faixa]
        if len(sub) < 5:
            print(f'  {faixa:<14} n={len(sub):>4} (insuficiente)')
            continue
        yt  = sub['label'].values
        yp  = sub['_pred'].values
        f1  = round(f1_score(yt, yp, average='macro', zero_division=0), 4)
        acc = round(accuracy_score(yt, yp), 4)
        n0  = (yt==0).sum()
        n1  = (yt==1).sum()
        print(f'  {faixa:<14} n={len(sub):>5} (0:{n0} 1:{n1})  f1={f1}  acc={acc}')
        registros_por_tamanho.append({
            'chave': chave, 'dataset_treino': ds_nome, 'modelo': res['mod_nome'],
            'faixa_tamanho': faixa, 'n_test': len(sub),
            'n_label0': int(n0), 'n_label1': int(n1),
            'f1_macro': f1, 'accuracy': acc,
        })

for chave, res in RESULTADOS.items():
    analisar_por_tamanho(chave, res)

## Seção 11 — Erros do melhor modelo (V4 balanced)

Seleciona o modelo V4 balanced com maior f1_macro e salva os erros com contexto.

In [ ]:
# Identificar melhor modelo V4 balanced
_chaves_v4 = [k for k in RESULTADOS if k.startswith('v4_balanced__')]
_melhor_v4 = max(_chaves_v4, key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])
print(f'Melhor V4 balanced: {_melhor_v4}')
print(f'  f1_macro = {RESULTADOS[_melhor_v4]["metricas"]["f1_macro"]}')

_res   = RESULTADOS[_melhor_v4]
_df    = DATASETS['v4_balanced']['df']
_split = DATASETS['v4_balanced']['split']

df_test_v4 = _df.loc[_split['idx_test']].copy()
df_test_v4['label_predito'] = _res['y_pred']
if _res['y_score'] is not None:
    df_test_v4['confianca'] = np.round(_res['y_score'], 4)
else:
    df_test_v4['confianca'] = None

df_erros_v4 = df_test_v4[df_test_v4['label'] != df_test_v4['label_predito']].copy()

_COLS_ERRO = [
    'texto_principal_modelo', 'label', 'label_predito', 'confianca',
    'dataset_origem', 'origem_qualidade', 'label_detalhe',
    'tema_query', 'subtema_query', 'url_origem',
]
_COLS_ERRO = [c for c in _COLS_ERRO if c in df_erros_v4.columns]
df_erros_v4 = df_erros_v4[_COLS_ERRO].copy()

print(f'Total de erros: {len(df_erros_v4)} de {len(df_test_v4)} ({100*len(df_erros_v4)/max(len(df_test_v4),1):.1f}%)')
if 'dataset_origem' in df_erros_v4.columns:
    print(f'Erros por dataset_origem:')
    print(df_erros_v4['dataset_origem'].value_counts().to_string())

# ── Erros melhor modelo V4 SC balanced ────────────────────────────────────
_chaves_sc = [k for k in RESULTADOS if k.startswith('v4_sc_balanced__')]
if _chaves_sc:
    _melhor_sc = max(_chaves_sc, key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])
    print(f'\nMelhor V4 SC balanced: {_melhor_sc}')
    print(f'  f1_macro = {RESULTADOS[_melhor_sc]["metricas"]["f1_macro"]}')
    _res_sc   = RESULTADOS[_melhor_sc]
    _df_sc    = DATASETS['v4_sc_balanced']['df']
    _split_sc = DATASETS['v4_sc_balanced']['split']
    _df_test_sc = _df_sc.loc[_split_sc['idx_test']].copy()
    _df_test_sc['label_predito'] = _res_sc['y_pred']
    df_erros_sc = _df_test_sc[_df_test_sc['label'] != _df_test_sc['label_predito']].copy()
    df_erros_sc = df_erros_sc[[c for c in _COLS_ERRO if c in df_erros_sc.columns]].copy()
    print(f'Total de erros SC: {len(df_erros_sc)} de {len(_df_test_sc)} ({100*len(df_erros_sc)/max(len(_df_test_sc),1):.1f}%)')
    if 'dataset_origem' in df_erros_sc.columns:
        print(df_erros_sc['dataset_origem'].value_counts().to_string())
else:
    df_erros_sc = pd.DataFrame()
    _melhor_sc  = None

## Seção 12 — Salvar relatórios CSV

In [ ]:
# ── métricas principais ──────────────────────────────────────────────────
_rows_met = []
for chave, res in RESULTADOS.items():
    m = res['metricas']
    _rows_met.append({
        'chave': chave,
        'dataset_treino': res['ds_nome'],
        'arquivo_dataset': DATASETS[res['ds_nome']]['path'].name,
        'modelo': res['mod_nome'],
        'total_registros': len(DATASETS[res['ds_nome']]['df']),
        'total_treino': len(DATASETS[res['ds_nome']]['split']['X_train']),
        'total_teste': len(DATASETS[res['ds_nome']]['split']['X_test']),
        'n_features_tfidf': res['n_feat'],
        'accuracy': m['accuracy'],
        'precision_macro': m['precision_macro'],
        'recall_macro': m['recall_macro'],
        'f1_macro': m['f1_macro'],
        'precision_label_0': m['precision_label_0'],
        'recall_label_0': m['recall_label_0'],
        'f1_label_0': m['f1_label_0'],
        'precision_label_1': m['precision_label_1'],
        'recall_label_1': m['recall_label_1'],
        'f1_label_1': m['f1_label_1'],
        'roc_auc': m['roc_auc'],
        'timestamp': TS,
    })

df_met = pd.DataFrame(_rows_met)
_ARQ_MET = DADOS_DIR / f'metricas_modelos_v4_source_control_{TS}.csv'
df_met.to_csv(_ARQ_MET, index=False, encoding='utf-8-sig')
print(f'Salvo: {_ARQ_MET.name}')

# ── por origem ───────────────────────────────────────────────────────────
if registros_por_origem:
    _ARQ_ORIG = DADOS_DIR / f'metricas_por_origem_v4_source_control_{TS}.csv'
    pd.DataFrame(registros_por_origem).to_csv(_ARQ_ORIG, index=False, encoding='utf-8-sig')
    print(f'Salvo: {_ARQ_ORIG.name}')
else:
    _ARQ_ORIG = None

# ── por qualidade ─────────────────────────────────────────────────────────
if registros_por_qualidade:
    _ARQ_QUAL = DADOS_DIR / f'metricas_por_qualidade_v4_source_control_{TS}.csv'
    pd.DataFrame(registros_por_qualidade).to_csv(_ARQ_QUAL, index=False, encoding='utf-8-sig')
    print(f'Salvo: {_ARQ_QUAL.name}')
else:
    _ARQ_QUAL = None

# ── por tamanho ───────────────────────────────────────────────────────────
if registros_por_tamanho:
    _ARQ_TAM = DADOS_DIR / f'metricas_por_tamanho_v4_source_control_{TS}.csv'
    pd.DataFrame(registros_por_tamanho).to_csv(_ARQ_TAM, index=False, encoding='utf-8-sig')
    print(f'Salvo: {_ARQ_TAM.name}')
else:
    _ARQ_TAM = None

# ── erros ─────────────────────────────────────────────────────────────────
_ARQ_ERROS = DADOS_DIR / f'erros_modelo_v4_source_control_{TS}.csv'
df_erros_v4.to_csv(_ARQ_ERROS, index=False, encoding='utf-8-sig')
print(f'Salvo: {_ARQ_ERROS.name}')
if not df_erros_sc.empty:
    _ARQ_ERROS_SC = DADOS_DIR / f'erros_modelo_v4_sc_{TS}.csv'
    df_erros_sc.to_csv(_ARQ_ERROS_SC, index=False, encoding='utf-8-sig')
    print(f'Salvo: {_ARQ_ERROS_SC.name}')
else:
    _ARQ_ERROS_SC = None

## Seção 13 — Salvar modelos (V4 balanced)

In [ ]:
NOMES_MODELO = {
    'v4_balanced__logreg':    'baseline_tfidf_logreg_v4_balanced',
    'v4_balanced__svm':       'baseline_tfidf_svm_v4_balanced',
    'v4_balanced__nb':        'baseline_tfidf_nb_v4_balanced',
    'v4_sc_balanced__logreg': 'baseline_tfidf_logreg_v4_sc_balanced',
    'v4_sc_balanced__svm':    'baseline_tfidf_svm_v4_sc_balanced',
    'v4_sc_balanced__nb':     'baseline_tfidf_nb_v4_sc_balanced',
}

_modelos_salvos = {}

for chave, nome_base in NOMES_MODELO.items():
    if chave not in RESULTADOS:
        continue
    res = RESULTADOS[chave]
    m   = res['metricas']
    df  = DATASETS[res['ds_nome']]['df']

    nome_ts     = f'{nome_base}_{TS}'
    path_joblib = MODELOS_DIR / f'{nome_ts}.joblib'
    path_meta   = MODELOS_DIR / f'{nome_ts}_meta.json'

    joblib.dump(res['pipe'], path_joblib)

    meta = {
        'nome': nome_ts,
        'timestamp': TS,
        'dataset_usado': DATASETS[res['ds_nome']]['path'].name,
        'n_registros': len(df),
        'distribuicao_label': df['label'].value_counts().to_dict(),
        'tfidf_params': TFIDF_PARAMS if res['mod_nome'] != 'nb' else TFIDF_PARAMS_NB,
        'modelo_params': {
            'logreg': {'solver':'lbfgs','class_weight':'balanced','max_iter':1000},
            'svm':    {'class_weight':'balanced','max_iter':5000,'calibrated':True},
            'nb':     {'alpha':1.0},
        }.get(res['mod_nome'], {}),
        'metricas': m,
        'n_features_tfidf': res['n_feat'],
        'candidato_producao': m['f1_macro'] >= 0.92,
        'observacao': (
            'V4 balanced/SC — base expandida com CheckAI Autoral + GFC. '
            'SC reduz dominancia academica. Avaliar f1 por dataset_origem '
            'antes de promover a producao.'
        ),
    }
    with open(path_meta, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

    _modelos_salvos[chave] = (path_joblib, path_meta)
    print(f'Salvo: {path_joblib.name}  (f1_macro={m["f1_macro"]})')

print(f'\n{len(_modelos_salvos)} modelos salvos (V4 balanced + V4 SC balanced).')

## Seção 14 — Relatório comparativo final

In [ ]:
# ── referências V3 (carrega métricas salvas se disponível) ───────────────
_v3_ref_files = sorted(DADOS_DIR.glob('metricas_modelos_v3_*.csv'))
V3_REF = {}
if _v3_ref_files:
    _df_v3_ref = pd.read_csv(_v3_ref_files[-1])
    for _, row in _df_v3_ref.iterrows():
        if row.get('dataset_treino') == 'v3_balanced':
            V3_REF[row['modelo']] = {
                'accuracy': row.get('accuracy'),
                'f1_macro': row.get('f1_macro'),
                'roc_auc':  row.get('roc_auc'),
            }

# ── métricas resumidas por chave ──────────────────────────────────────────
def _m(chave):
    return RESULTADOS[chave]['metricas'] if chave in RESULTADOS else {}

# Melhor por dataset
def _melhor(prefixo):
    chs = [k for k in RESULTADOS if k.startswith(prefixo)]
    return max(chs, key=lambda k: RESULTADOS[k]['metricas']['f1_macro']) if chs else None

_mel_v3   = _melhor('v3_balanced__')
_mel_v4b  = _melhor('v4_balanced__')
_mel_v4t  = _melhor('v4_text_control__')
_mel_v4sc = _melhor('v4_sc_balanced__')
_mel_v4st = _melhor('v4_sc_text_control__')

# Alertas
def _gap_aviso(val, ref, thr=0.05):
    return f'GAP {ref-val:.4f}' if val is not None and ref is not None and ref - val > thr else 'OK'

_f1_v3   = _m(_mel_v3).get('f1_macro', 0)   if _mel_v3   else 0
_f1_v4b  = _m(_mel_v4b).get('f1_macro', 0)  if _mel_v4b  else 0
_f1_v4t  = _m(_mel_v4t).get('f1_macro', 0)  if _mel_v4t  else 0
_f1_v4sc = _m(_mel_v4sc).get('f1_macro', 0) if _mel_v4sc else 0
_f1_v4st = _m(_mel_v4st).get('f1_macro', 0) if _mel_v4st else 0

# f1 na base própria (CHECKAI_AUTORAL + CHECKAI_PROPRIO_GFC)
_PROPRIA = {'CHECKAI_AUTORAL', 'CHECKAI_PROPRIO_GFC', 'V2_PROPRIA'}
def _f1_propria(chave):
    if chave not in RESULTADOS:
        return None
    res    = RESULTADOS[chave]
    df_    = DATASETS[res['ds_nome']]['df']
    split_ = DATASETS[res['ds_nome']]['split']
    if 'dataset_origem' not in df_.columns:
        return None
    df_t = df_.loc[split_['idx_test']].copy()
    df_t['_pred'] = res['y_pred']
    sub  = df_t[df_t['dataset_origem'].isin(_PROPRIA)]
    if len(sub) < 10:
        return None
    return round(f1_score(sub['label'].values, sub['_pred'].values, average='macro', zero_division=0), 4)

_f1p_v3   = _f1_propria(_mel_v3)   if _mel_v3   else None
_f1p_v4b  = _f1_propria(_mel_v4b)  if _mel_v4b  else None
_f1p_v4sc = _f1_propria(_mel_v4sc) if _mel_v4sc else None

SEP = '=' * 72
print(SEP)
print('RELATÓRIO FINAL — TREINO BASELINE V4 COMPARATIVO')
print(SEP)

print('\n[Datasets utilizados]')
for nome, info in DATASETS.items():
    df_ = info['df']
    print(f'  {nome:<22}: {info["path"].name}  ({len(df_)} registros)')

print('\n[Tabela geral de métricas]')
print(f"  {'Dataset/Modelo':<34} {'acc':>6} {'f1_mac':>7} {'f1_0':>6} {'f1_1':>6} {'roc':>7}")
print('  ' + '-' * 70)
for chave in sorted(RESULTADOS.keys()):
    m_ = _m(chave)
    roc_ = f"{m_.get('roc_auc','N/A'):.4f}" if isinstance(m_.get('roc_auc'), float) else 'N/A'
    print(f"  {chave:<34} {m_.get('accuracy',0):>6.4f} {m_.get('f1_macro',0):>7.4f} "
          f"{m_.get('f1_label_0',0):>6.4f} {m_.get('f1_label_1',0):>6.4f} {roc_:>7}")

print('\n[Melhor modelo por dataset]')
for prefixo, mel in [('v3_balanced', _mel_v3), ('v4_balanced', _mel_v4b), ('v4_text_control', _mel_v4t)]:
    if mel:
        m_ = _m(mel)
        print(f'  {mel:<34} f1={m_.get("f1_macro")}  roc={m_.get("roc_auc")}')

print('\n[Comparação V3 / V4 / V4 Source Control]')
print(f'  V3 balanced          : f1={_f1_v3}  (baseline historico)')
print(f'  V4 balanced          : f1={_f1_v4b}  (delta vs V3={_f1_v4b-_f1_v3:+.4f})')
print(f'  V4 text_control      : f1={_f1_v4t}  (delta vs V3={_f1_v4t-_f1_v3:+.4f})')
print(f'  V4 SC balanced       : f1={_f1_v4sc}  (delta vs V4={_f1_v4sc-_f1_v4b:+.4f})')
print(f'  V4 SC text_control   : f1={_f1_v4st}  (delta vs V4={_f1_v4st-_f1_v4b:+.4f})')

_bias_sz  = abs(_f1_v4b  - _f1_v4t)  > 0.02
_bias_sz_sc = abs(_f1_v4sc - _f1_v4st) > 0.02
_bias_orig = abs(_f1_v4b - _f1_v4sc) > 0.03
_bias_msg = f'V4_bal vs TC={_f1_v4b-_f1_v4t:.4f} | SC_bal vs SC_TC={_f1_v4sc-_f1_v4st:.4f}'
print(f'  Viés tamanho: {_bias_msg}')
_orig_msg = f'V4_bal vs V4_SC_bal={_f1_v4b-_f1_v4sc:.4f} -- possivel vies de origem' if _bias_orig else f'V4_bal vs V4_SC_bal={_f1_v4b-_f1_v4sc:.4f} -- origem sob controle'
print(f'  Viés origem : {_orig_msg}')

print('\n[Base propria -- f1_macro (CHECKAI_AUTORAL + CHECKAI_PROPRIO_GFC + V2_PROPRIA)]')
print(f'  V3 balanced    : f1_propria = {_f1p_v3}')
print(f'  V4 balanced    : f1_propria = {_f1p_v4b}')
print(f'  V4 SC balanced : f1_propria = {_f1p_v4sc}')
if _f1p_v4b and _f1p_v4sc:
    _delta_sc_prop = _f1p_v4sc - _f1p_v4b if _f1p_v4b else None
    _sc_improve = 'MELHORA' if _delta_sc_prop and _delta_sc_prop > 0 else 'PIORA/IGUAL'
    print(f'  SC vs V4 base propria: {_delta_sc_prop:+.4f} ({_sc_improve})')

## Seção 15 — Alertas metodológicos e geração do relatório .md

In [ ]:
# ── construir linhas do relatório ────────────────────────────────────────
_L = []
_L.append('# Relatório — Treino Baseline V4 Comparativo')
_L.append('')
_L.append(f'**Data:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
_L.append(f'**Timestamp:** {TS}')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 1. Arquivos usados')
_L.append('')
for nome, info in DATASETS.items():
    _L.append(f'- {nome}: `{info["path"].name}`')
_L.append('')
_L.append('## 2. Datasets avaliados')
_L.append('')
for nome, info in DATASETS.items():
    df_ = info['df']
    _L.append(f'- **{nome}**: {len(df_)} registros | label=0:{(df_["label"]==0).sum()} label=1:{(df_["label"]==1).sum()}')
_L.append('')
_L.append('## 3. Modelos treinados')
_L.append('')
_L.append('- Logistic Regression (lbfgs, class_weight=balanced)')
_L.append('- Linear SVM (CalibratedClassifierCV, class_weight=balanced)')
_L.append('- MultinomialNB')
_L.append('')
_L.append('## 4. Tabela geral de métricas')
_L.append('')
_L.append('| Dataset/Modelo | accuracy | f1_macro | f1_0 | f1_1 | roc_auc |')
_L.append('|---|---|---|---|---|---|')
for chave in sorted(RESULTADOS.keys()):
    m_ = _m(chave)
    roc_ = f"{m_.get('roc_auc','N/A'):.4f}" if isinstance(m_.get('roc_auc'), float) else 'N/A'
    _L.append(f'| {chave} | {m_.get("accuracy",0):.4f} | {m_.get("f1_macro",0):.4f} | '
              f'{m_.get("f1_label_0",0):.4f} | {m_.get("f1_label_1",0):.4f} | {roc_} |')
_L.append('')
_L.append('## 5. Melhor modelo geral')
_L.append('')
_mel_geral = max(RESULTADOS.keys(), key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])
_L.append(f'- **{_mel_geral}**  f1_macro={_m(_mel_geral).get("f1_macro")}  roc_auc={_m(_mel_geral).get("roc_auc")}')
_L.append('')
_L.append('## 6. Melhor modelo metodológico')
_L.append('')
_L.append(f'- Selecionado: **{_mel_v4b}** (melhor V4 balanced por f1_macro e base própria)')
_L.append('')
_L.append('## 7. Comparação V3 vs V4')
_L.append('')
_L.append('| Versão | Melhor modelo | f1_macro | delta |')
_L.append('|---|---|---|---|')
_L.append(f'| V3 balanced | {_mel_v3} | {_f1_v3:.4f} | — |')
_L.append(f'| V4 balanced | {_mel_v4b} | {_f1_v4b:.4f} | {_f1_v4b-_f1_v3:+.4f} |')
_L.append(f'| V4 text_control | {_mel_v4t} | {_f1_v4t:.4f} | {_f1_v4t-_f1_v3:+.4f} |')
_L.append('')
_L.append('## 8. Avaliação da base própria')
_L.append('')
_L.append(f'- f1_macro base própria V3 balanced : {_f1p_v3}')
_L.append(f'- f1_macro base própria V4 balanced : {_f1p_v4b}')
if _f1p_v3 and _f1p_v4b:
    _L.append(f'- Melhora: {_f1p_v4b-_f1p_v3:+.4f}')
    if _f1p_v4b < _f1_v4b - 0.05:
        _L.append(f'- ALERTA: gap de generalização — base própria ({_f1p_v4b:.4f}) vs geral ({_f1_v4b:.4f})')
_L.append('')
_L.append('## 9. Avaliação por origem_qualidade')
_L.append('')
if registros_por_qualidade:
    _df_q = pd.DataFrame(registros_por_qualidade)
    _q_v4 = _df_q[(_df_q['chave'] == _mel_v4b)]
    _L.append('```')
    _L.append(_q_v4[['origem_qualidade','n_test','f1_macro','accuracy']].to_string(index=False))
    _L.append('```')
_L.append('')
_L.append('## 10. Avaliação por faixa de tamanho')
_L.append('')
if registros_por_tamanho:
    _df_t = pd.DataFrame(registros_por_tamanho)
    for ch_t in [_mel_v4b, _mel_v4t]:
        if ch_t is None:
            continue
        _sub = _df_t[_df_t['chave'] == ch_t]
        _L.append(f'### {ch_t}')
        _L.append('```')
        _L.append(_sub[['faixa_tamanho','n_test','f1_macro','accuracy']].to_string(index=False))
        _L.append('```')
        _L.append('')
_L.append('')
_L.append('## 11. Principais erros')
_L.append('')
_L.append(f'- Total de erros V4 balanced melhor: {len(df_erros_v4)}')
if 'dataset_origem' in df_erros_v4.columns:
    _L.append('- Por dataset_origem:')
    for orig, cnt in df_erros_v4['dataset_origem'].value_counts().items():
        _L.append(f'  - {orig}: {cnt}')
_L.append('')
_L.append('## 7b. Comparacao V4 balanced vs V4 source_control_balanced')
_L.append('')
_L.append('| Versao | melhor modelo | f1_macro | delta vs V3 | delta vs V4 bal |')
_L.append('|---|---|---|---|---|')
for _k, _f, _dv3, _dv4 in [
    ('V3 balanced',       _mel_v3,   0,               None),
    ('V4 balanced',       _mel_v4b,  _f1_v4b-_f1_v3,  None),
    ('V4 text_control',   _mel_v4t,  _f1_v4t-_f1_v3,  _f1_v4t-_f1_v4b),
    ('V4 SC balanced',    _mel_v4sc, _f1_v4sc-_f1_v3, _f1_v4sc-_f1_v4b),
    ('V4 SC text_control',_mel_v4st, _f1_v4st-_f1_v3, _f1_v4st-_f1_v4b),
]:
    _f1_ = _m(_k).get('f1_macro', 0) if _k else 0
    _dv3_ = f'{_dv3:+.4f}' if _dv3 is not None else '--'
    _dv4_ = f'{_dv4:+.4f}' if _dv4 is not None else '--'
    _L.append(f'| {_f} or N/A | {_f} | {_f1_:.4f} | {_dv3_} | {_dv4_} |')
_L.append('')
_L.append('### Base propria (CHECKAI_AUTORAL + GFC + V2_PROPRIA)')
_L.append('')
_L.append('| Versao | f1_propria |')
_L.append('|---|---|')
_L.append(f'| V3 balanced | {_f1p_v3} |')
_L.append(f'| V4 balanced | {_f1p_v4b} |')
_L.append(f'| V4 SC balanced | {_f1p_v4sc} |')
if _f1p_v4b and _f1p_v4sc:
    _delta_prop = _f1p_v4sc - _f1p_v4b
    _L.append(f'| Delta SC vs V4 | {_delta_prop:+.4f} |')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 7c. Analise de vies de origem')
_L.append('')
_L.append(f'- V4_bal vs V4_SC_bal (delta f1): {_f1_v4b-_f1_v4sc:.4f}')
_L.append(f'- Interpretacao: {"SC melhorou generalização" if _f1_v4sc >= _f1_v4b else "SC reduziu metrica geral (esperado se base menor)"}' )
_L.append(f'- V4_bal vs V4_TC: {_f1_v4b-_f1_v4t:.4f} | SC_bal vs SC_TC: {_f1_v4sc-_f1_v4st:.4f}')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## 12. Alertas metodológicos')
_L.append('')
_al = []
_acad_pct = 100*(df_v4_balanced['dataset_origem'].isin({'FAKEBR_TEXT_NORMALIZED','FAKETRUEBR'})).sum()/max(len(df_v4_balanced),1) if 'dataset_origem' in df_v4_balanced.columns else 0
if _acad_pct > 70:
    _al.append(f'Base acadêmica ainda domina V4 balanced ({_acad_pct:.0f}%)')
if _f1p_v4b and _f1p_v4b < _f1_v4b - 0.05:
    _al.append(f'Gap base própria: f1_geral={_f1_v4b:.4f} vs f1_propria={_f1p_v4b:.4f} ({_f1_v4b-_f1p_v4b:.4f})')
if _bias_sz:
    _al.append(f'Possível viés de tamanho: V4_bal={_f1_v4b:.4f} vs V4_tc={_f1_v4t:.4f} (delta={_f1_v4b-_f1_v4t:.4f})')
if _f1_v4b > _f1_v3:
    _al.append(f'V4 melhorou sobre V3 (delta={_f1_v4b-_f1_v3:+.4f})')
else:
    _al.append(f'V4 NÃO melhorou sobre V3 (delta={_f1_v4b-_f1_v3:+.4f})')
for a in _al:
    _L.append(f'- {a}')
_L.append('')
_L.append('## 13. Recomendação final')
_L.append('')
_L.append('| Uso | Dataset | Modelo |')
_L.append('|---|---|---|')
_L.append(f'| **Treino principal** | V4 balanced | {_mel_v4b} |')
_L.append(f'| Ablação viés tamanho | V4 text_control | {_mel_v4t} |')
_L.append(f'| Referência histórica | V3 balanced | {_mel_v3} |')
_L.append('')
_L.append('---')
_L.append('')
_L.append('## Conclusão')
_L.append('')
_L.append('O treino V4 avalia se a expansão da base própria do CheckAI melhora a generalização '
          'do classificador. A comparação entre V3 balanced, V4 balanced e V4 text_control '
          'permite separar ganho real de dados de possível viés de tamanho ou origem.')
_L.append('')
_L.append('---')
_L.append(f'*Gerado por `src/treino_baseline_v4_comparativo.ipynb` | timestamp `{TS}`*')

relatorio = '\n'.join(_L)
_ARQ_REL = DADOS_DIR / f'relatorio_treino_baseline_v4_{TS}.md'
with open(_ARQ_REL, 'w', encoding='utf-8') as f_rel:
    f_rel.write(relatorio)
print(f'Relatório salvo: {_ARQ_REL.name}')

## Seção 16 — Resumo final

In [ ]:
SEP = '=' * 72
print(SEP)
print('RESUMO — TREINO BASELINE V4 + SOURCE CONTROL COMPARATIVO')
print(SEP)
print()
print(f'Melhor modelo por f1_macro  : {_mel_geral}  f1={_m(_mel_geral).get("f1_macro")}')
print(f'Melhor metodológico (V4 bal): {_mel_v4b}  f1={_f1_v4b}')
print()
print(f'f1_macro V3 balanced          : {_f1_v3}')
print(f'f1_macro V4 balanced          : {_f1_v4b}  (delta vs V3={_f1_v4b-_f1_v3:+.4f})')
print(f'f1_macro V4 text_control      : {_f1_v4t}  (delta vs V3={_f1_v4t-_f1_v3:+.4f})')
print(f'f1_macro V4 SC balanced       : {_f1_v4sc}  (delta vs V4={_f1_v4sc-_f1_v4b:+.4f})')
print(f'f1_macro V4 SC text_control   : {_f1_v4st}  (delta vs V4={_f1_v4st-_f1_v4b:+.4f})')
print()
print(f'f1_macro base propria V3     : {_f1p_v3}')
print(f'f1_macro base propria V4     : {_f1p_v4b}')
print(f'f1_macro base propria V4 SC  : {_f1p_v4sc}')
print()
_usar_v4 = _f1_v4b >= _f1_v3 and (_f1p_v4b is None or _f1p_v4b >= 0.80)
_sc_util = _f1p_v4sc is None or _f1p_v4sc >= (_f1p_v4b or 0)
print(f'Usar V4 como dataset principal?  {"SIM" if _usar_v4 else "AVALIAR"}')
print(f'SC balanced util para ablacao?   {"SIM" if _sc_util else "VERIFICAR -- SC piorou base propria"}')
if not _usar_v4:
    print('  -> Revisar gap base propria antes de substituir V3.')
print()
print('Proxima etapa: fechar versao metodologica final e preparar TCC.')
print()
print('Arquivos gerados:')
print(f'  {_ARQ_MET.name}')
if _ARQ_ORIG: print(f'  {_ARQ_ORIG.name}')
if _ARQ_QUAL: print(f'  {_ARQ_QUAL.name}')
if _ARQ_TAM:  print(f'  {_ARQ_TAM.name}')
print(f'  {_ARQ_ERROS.name}')
print(f'  {_ARQ_REL.name}')
for ch, (pj, pm) in _modelos_salvos.items():
    print(f'  {pj.name}')
    print(f'  {pm.name}')
print(SEP)